In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, dense_rank, avg, when

spark = SparkSession.builder.appName("EmployeeRanking").getOrCreate()

data = [
(1, "Ravi Kumar", "Engineering", "Hyderabad", 92000),
(2, "Sneha Reddy", "Engineering", "Bengaluru", 97000),
(3, "Amit Shah", "Finance", "Mumbai", 88000),
(4, "Pooja Nair", "HR", "Chennai", 69000),
(5, "Nikhil Verma", "Engineering", "Delhi", 85000),
(6, "Meera Iyer", "Finance", "Hyderabad", 91000),
(7, "Karan Malhotra", "Marketing", "Mumbai", 76000),
(8, "Anjali Rao", "HR", "Bengaluru", 72000),
(9, "Vivek Joshi", "Finance", "Delhi", 94000),
(10, "Fatima Khan", "Marketing", "Chennai", 81000),
(11, "Aditya Menon", "Engineering", "Mumbai", 99000),
(12, "Lakshmi Das", "Finance", "Bengaluru", 87000)
]

columns = ["emp_id", "emp_name", "department", "city", "salary"]
df = spark.createDataFrame(data, columns)
window_spec = Window.partitionBy("department").orderBy(df.salary.desc())
df = df.withColumn("rank", rank().over(window_spec))
df = df.withColumn("salary_level",when(df.salary >= 95000, "Very High").when(df.salary >= 85000, "High").otherwise("Standard"))
top2 = df.withColumn("dense_rank",dense_rank().over(window_spec)).filter("dense_rank <= 2")
top2.show()
df.groupBy("department").agg(avg("salary").alias("average_salary")).show()

+------+--------------+-----------+---------+------+----+------------+----------+
|emp_id|      emp_name| department|     city|salary|rank|salary_level|dense_rank|
+------+--------------+-----------+---------+------+----+------------+----------+
|    11|  Aditya Menon|Engineering|   Mumbai| 99000|   1|   Very High|         1|
|     2|   Sneha Reddy|Engineering|Bengaluru| 97000|   2|   Very High|         2|
|     9|   Vivek Joshi|    Finance|    Delhi| 94000|   1|        High|         1|
|     6|    Meera Iyer|    Finance|Hyderabad| 91000|   2|        High|         2|
|     8|    Anjali Rao|         HR|Bengaluru| 72000|   1|    Standard|         1|
|     4|    Pooja Nair|         HR|  Chennai| 69000|   2|    Standard|         2|
|    10|   Fatima Khan|  Marketing|  Chennai| 81000|   1|    Standard|         1|
|     7|Karan Malhotra|  Marketing|   Mumbai| 76000|   2|    Standard|         2|
+------+--------------+-----------+---------+------+----+------------+----------+

+-----------+--